In [ ]:
#Upgrade all
!pip install -U transformers torch accelerate
!pip install -U jinja2
!pip install --upgrade gradio

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(
    device
)
model.eval()

print("Model loaded successfully!")

# System instruction
SYSTEM_PROMPT = {
    "role": "system",
    "content": "You are a concise, helpful assistant for NIC e-Governance demos."
}

# Gradio ChatInterface callback
def ask_bot(user_message, history):
    messages = [
        {
            "role": "system",
            "content": "You are a concise, helpful assistant for NIC e-Governance demos."
        }
    ]

    # Process the latest 6 exchanges
    for item in history[-12:]:

        # Handle dictionary-based history
        if isinstance(item, dict):

            role = item.get("role")
            content = item.get("content")

            # Extract text from list-based content
            if isinstance(content, list):
                text_parts = []

                for part in content:
                    if isinstance(part, str):
                        text_parts.append(part)

                    elif isinstance(part, dict):
                        if part.get("type") == "text":
                            text_parts.append(part.get("text", ""))

                content = " ".join(text_parts)

            # Add only valid text messages
            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })

    # Add current user message
    messages.append({
        "role": "user",
        "content": str(user_message)
    })

    # Create model prompt
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(device)

    # Generate response
    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract generated tokens
    generated_ids = output_ids[0][
        inputs["input_ids"].shape[1]:
    ]

    reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    print("Received history:", history)

    return reply

print("Chatbot ready!")

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=ask_bot,
    title="NIC e-Governance AI Assistant",
    description="AI-powered conversational assistant using Qwen 2.5",
    examples=[
        "What is e-Governance?",
        "Explain the role of NIC.",
        "How can AI improve citizen services?"
    ]
)

demo.launch(share=True)